두 데이터 불러오기

In [ ]:
import pandas as pd

# BC 소비데이터 — 코드값 컬럼은 문자열로 고정해야 조인할 때 타입이 맞는다
bc = pd.read_csv("data/ABP_CONTEST_DATA.csv", encoding="utf-8",
                 dtype={"STRD_YYMM": str, "GENDER_CD": str,
                        "AGE_CD": str, "TP_BUZ_NO": str})

# 인구 최종본 — population.ipynb에서 만든 파일
# encoding="utf-8-sig" : 저장할 때 BOM을 붙였으므로 읽을 때도 맞춰 준다
pop = pd.read_csv("data/인구_시군구_성별_연령_202601_202606.csv", encoding="utf-8-sig",
                  dtype={"STRD_YYMM": str, "GENDER_CD": str,
                         "AGE_CD": str, "행정구역코드": str})

print("BC :", bc.shape)   # (242574, 9)
print("인구:", pop.shape)  # (18360, 6)
print()
print(bc.head(3).to_string())
print()
print(pop.head(3).to_string())


BC 쪽에 인구와 같은 형식의 지역 키 만들기

In [ ]:
# 두 컬럼을 공백으로 이어 붙여 인구 데이터의 '행정구역명'과 같은 형식으로 만든다
#  예) "경기도" + "수원시 팔달구" → "경기도 수원시 팔달구"
bc["행정구역명"] = bc["SIDO_NM"].str.strip() + " " + bc["CCG_NM"].str.strip()

# 세종시는 BC에서 "세종특별자치시 세종특별자치시"가 되므로 인구 쪽 표기로 통일
bc["행정구역명"] = bc["행정구역명"].replace("세종특별자치시 세종특별자치시",
                                      "세종특별자치시")

# 혹시 공백이 두 칸 이상 들어간 경우를 대비해 하나로 정리
bc["행정구역명"] = bc["행정구역명"].str.replace(r"\s+", " ", regex=True).str.strip()

# 양쪽 지역 집합을 비교 — 한쪽에만 있는 지역이 있으면 조인에서 누락된다
bc_지역 = set(bc["행정구역명"])
pop_지역 = set(pop["행정구역명"])

print("BC 지역 수  :", len(bc_지역))
print("인구 지역 수:", len(pop_지역))
print("공통        :", len(bc_지역 & pop_지역))
print()
print("BC에만 있음  :", sorted(bc_지역 - pop_지역))
print("인구에만 있음:", sorted(pop_지역 - bc_지역))


내국인 개인만 필터링

In [ ]:
# 기획서 2단계: 법인 제외, 개인 소비만 사용
#  - GENDER_CD 1(남)·2(여) = 내국인 개인   ← 이것만 남긴다
#  - GENDER_CD 3 = 외국인 (연령대는 1~6을 정상적으로 가짐! 자동 제외되지 않는다)
#  - GENDER_CD x = 법인 (AGE_CD도 x)
#
# 분모가 주민등록인구(내국인)이므로 분자도 내국인으로 맞춰야 1인당 소비가 왜곡되지 않는다
bc_개인 = bc[bc["GENDER_CD"].isin(["1", "2"])].copy()

print("필터 전:", len(bc), "행 /", f"{bc['amt'].sum()/1e12:.2f}조")
print("필터 후:", len(bc_개인), "행 /", f"{bc_개인['amt'].sum()/1e12:.2f}조")
print("제외 비중:", f"{(1 - bc_개인['amt'].sum()/bc['amt'].sum())*100:.1f}%")
print()

# 남은 값이 의도한 조합뿐인지 확인 (AGE_CD에 'x'가 없어야 정상)
print("남은 GENDER_CD:", sorted(bc_개인["GENDER_CD"].unique()))
print("남은 AGE_CD   :", sorted(bc_개인["AGE_CD"].unique()))

# 3단계(외국인 포함 재합산)에서 쓸 수 있게 외국인 데이터도 따로 보관해 둔다
bc_외국인 = bc[bc["GENDER_CD"] == "3"].copy()
print()
print("외국인 별도 보관:", len(bc_외국인), "행 /", f"{bc_외국인['amt'].sum()/1e12:.2f}조")


시군구×업종 집계 + 인구 조인 + 1인당 소비액

In [ ]:
from IPython.display import display

# 숫자를 천단위 쉼표 + 소수점 1자리로 통일 — 노트북 전체에 적용된다
pd.set_option("display.float_format", lambda x: f"{x:,.1f}")

성인 = ["2", "3", "4", "5", "6"]   # 0~19세(AGE_CD 1)는 분자·분모에서 함께 제외

# 1) 분자: 내국인 개인 × 20대 이상 → 시군구×업종별 6개월 합계
bc_screen = bc[(bc["GENDER_CD"].isin(["1", "2"])) & (bc["AGE_CD"].isin(성인))]

소비 = bc_screen.groupby(["행정구역명", "TP_BUZ_NO", "TP_BUZ_NM"], as_index=False).agg(
    amt=("amt", "sum"),   # 6개월 총 이용금액
    cnt=("cnt", "sum"),   # 6개월 총 이용건수
)

# 2) 분모: 20대 이상 인구 (월별 합산 → 6개월 평균)
인구 = (pop[pop["AGE_CD"].isin(성인)]
       .groupby(["행정구역명", "STRD_YYMM"])["인구"].sum()
       .groupby("행정구역명").mean()
       .rename("성인인구").reset_index())

# 3) 조인
screen = 소비.merge(인구, on="행정구역명", how="left")
print(f"조합 수: {len(screen)}  |  인구 결측: {screen['성인인구'].isna().sum()}건")

# 4) 1인당 월평균 소비액
screen["1인당월소비"] = screen["amt"] / 6 / screen["성인인구"]

# 5) 업종별 전국 평균 (같은 업종끼리 비교해야 의미가 있다)
전국평균 = bc_screen.groupby("TP_BUZ_NO")["amt"].sum() / 6 / 인구["성인인구"].sum()
screen["전국평균"] = screen["TP_BUZ_NO"].map(전국평균)

# 6) 기획서 핵심 지표 — "기대치의 몇 %인가"
screen["기대치대비%"] = screen["1인당월소비"] / screen["전국평균"] * 100

print(f"전국 성인인구: {인구['성인인구'].sum():,.0f}명")


In [ ]:
# 업종별 전국 1인당 월소비 — 표로 출력
업종표 = (bc_screen.groupby(["TP_BUZ_NO", "TP_BUZ_NM"], as_index=False)["amt"].sum()
        .assign(전국1인당월소비=lambda d: d["amt"] / 6 / 인구["성인인구"].sum(),
                금액_조=lambda d: d["amt"] / 1e12)
        .drop(columns="amt")
        .sort_values("전국1인당월소비", ascending=False))

display(업종표)


In [ ]:
# 기대치 대비 낮은 순 상위 12개
후보 = (screen.sort_values("기대치대비%")
       .head(12)[["행정구역명", "TP_BUZ_NM", "amt", "1인당월소비", "전국평균", "기대치대비%"]]
       .assign(금액_억=lambda d: d["amt"] / 1e8)      # 억 단위가 읽기 편하다
       .drop(columns="amt"))

display(후보)


스크리닝 — 소형 업종 제외 + 업종 내 시장규모 상위 50%

In [ ]:
# 1) 커버리지가 낮아 비율이 불안정한 업종 제외
#    갈비전문점(8002): 105개 시군구만, 전국 1인당 월 33원
#    한정식(8003)   : 39개 시군구만, 전국 1인당 월 5원
제외업종 = ["8002", "8003"]
screen9 = screen[~screen["TP_BUZ_NO"].isin(제외업종)].copy()

# 2) 업종 내 시장규모 상위 50%만 남긴다
#    transform("median") : 각 행에 '그 행이 속한 업종의 중앙값'을 붙여 준다
#    업종마다 규모가 10배 이상 차이나므로, 전체 기준이 아니라 업종 안에서 비교해야 공평하다
screen9["업종내중앙값"] = screen9.groupby("TP_BUZ_NO")["amt"].transform("median")
후보풀 = screen9[screen9["amt"] >= screen9["업종내중앙값"]].copy()

print(f"제외 후 조합: {len(screen9)}  →  상위 50% 후보풀: {len(후보풀)}")


In [ ]:
# 업종별로 후보풀이 몇 개씩 남았는지 확인
display(후보풀.groupby("TP_BUZ_NM", as_index=False)
        .agg(후보수=("amt", "size"), 최소금액_억=("amt", lambda x: x.min() / 1e8))
        .sort_values("후보수", ascending=False))


In [ ]:
# 기획서 2단계 결과 — 저침투 의심 후보 상위 15개
후보15 = (후보풀.sort_values("기대치대비%").head(15)
        .assign(금액_억=lambda d: d["amt"] / 1e8)
        [["행정구역명", "TP_BUZ_NM", "금액_억", "1인당월소비", "전국평균", "기대치대비%"]])

display(후보15)


업종 특화지수로 전환

In [ ]:
# 스크리닝 대상: 내국인 개인 × 20대 이상 × 9개 업종
bc_mix = bc[(bc["GENDER_CD"].isin(["1", "2"])) &
            (bc["AGE_CD"].isin(성인)) &
            (~bc["TP_BUZ_NO"].isin(제외업종))]

mix = bc_mix.groupby(["행정구역명", "TP_BUZ_NO", "TP_BUZ_NM"], as_index=False)["amt"].sum()

# 1) 지역 안에서의 업종 비중
#    transform("sum") : 각 행에 '그 지역의 전체 BC 소비액'을 붙여 준다
mix["지역총액"] = mix.groupby("행정구역명")["amt"].transform("sum")
mix["지역내비중"] = mix["amt"] / mix["지역총액"]

# 2) 전국 업종 비중 (비교 기준)
전국비중 = bc_mix.groupby("TP_BUZ_NO")["amt"].sum() / bc_mix["amt"].sum()
mix["전국비중"] = mix["TP_BUZ_NO"].map(전국비중)

# 3) 특화지수 = 지역 비중 ÷ 전국 비중 × 100
#    100 = 전국 평균과 같은 구성, 100 미만 = 그 지역에서 이 업종이 전국 대비 과소
#    유동인구가 많아 모든 업종 금액이 함께 커져도 '비중'은 변하지 않으므로
#    도심/베드타운 차이가 자동으로 상쇄된다
mix["특화지수"] = mix["지역내비중"] / mix["전국비중"] * 100

display(pd.DataFrame({"전국비중_%": (전국비중 * 100).round(1)})
        .join(mix.groupby("TP_BUZ_NO")["TP_BUZ_NM"].first()))


In [ ]:
# 시장규모 상위 50% 필터는 그대로 유지 (규모 없는 조합은 공략 가치가 없다)
mix["업종내중앙값"] = mix.groupby("TP_BUZ_NO")["amt"].transform("median")
후보풀2 = mix[mix["amt"] >= mix["업종내중앙값"]].copy()

후보15 = (후보풀2.sort_values("특화지수").head(15)
        .assign(금액_억=lambda d: d["amt"] / 1e8,
                지역내비중_pct=lambda d: d["지역내비중"] * 100,
                전국비중_pct=lambda d: d["전국비중"] * 100)
        [["행정구역명", "TP_BUZ_NM", "금액_억", "지역내비중_pct", "전국비중_pct", "특화지수"]])

display(후보15)


두 지표 교차 — 최종 후보

In [ ]:
# 두 지표는 약점이 서로 다르다
#  - 기대치대비 : 유동인구가 많은 도심이 높게 나오는 약점
#  - 특화지수   : 한 업종이 지배하면 나머지가 눌리는 약점(부산 북구 대형할인점 50.7%)
# → 둘 다 낮은 조합만 남기면 각자의 약점이 상쇄된다

# 앞서 만든 screen(기대치대비)과 mix(특화지수)를 하나로 합친다
지표 = mix.merge(
    screen[["행정구역명", "TP_BUZ_NO", "1인당월소비", "기대치대비%"]],
    on=["행정구역명", "TP_BUZ_NO"], how="left"
)

# 시장규모 상위 50% 필터 유지
지표["업종내중앙값"] = 지표.groupby("TP_BUZ_NO")["amt"].transform("median")
후보풀3 = 지표[지표["amt"] >= 지표["업종내중앙값"]].copy()

# 두 지표 모두 전국 평균(100) 미만인 조합만
저침투 = 후보풀3[(후보풀3["기대치대비%"] < 100) & (후보풀3["특화지수"] < 100)].copy()

# 종합점수 = 두 지표의 기하평균
#  곱한 뒤 제곱근 → 한쪽만 극단적으로 낮은 경우보다 '둘 다 낮은' 경우를 앞세운다
저침투["종합점수"] = (저침투["기대치대비%"] * 저침투["특화지수"]) ** 0.5

print(f"후보풀: {len(후보풀3)}  →  두 지표 모두 100 미만: {len(저침투)}")


In [ ]:
# 3단계 외부검증으로 넘길 최종 후보 15개
최종후보 = (저침투.sort_values("종합점수").head(15)
         .assign(금액_억=lambda d: d["amt"] / 1e8)
         [["행정구역명", "TP_BUZ_NM", "금액_억", "기대치대비%", "특화지수", "종합점수"]])

display(최종후보)


업종 매핑 정의

In [ ]:
import requests, time, os
from api_key import DATA_GO_KR_KEY      # code/api_key.py (git 제외됨)

# BC 업종 → 소상공인 상가정보 업종코드
# 대회 데이터 설명서(TPBUZ_DT_NM)를 근거로 매칭했다
업종매핑 = {
    "4010": ["G20405"],                                    # 편의점
    "4020": ["G20404"],                                    # 슈퍼마켓
    "8001": ["I201"],                                      # 일반한식
    "8004": ["I203"],                                      # 일식회집 (횟집·초밥·참치)
    "8005": ["I202"],                                      # 중국음식
    # 서양음식: 설명서에 카페·커피·피자·햄버거·토스트·주스가 모두 포함돼 있다
    "8006": ["I204", "I212", "I21003", "I21004", "I21005"],
    # 스넥: 분식·김밥·만두·치킨 (피자·버거는 설명서상 서양음식으로 감)
    "8021": ["I21007", "I21006"],
    # 제과점: 설명서에 떡·한과·아이스크림이 명시돼 있다
    "8301": ["I21001", "I21002", "I21008"],
    # 4004 대형할인점은 상가정보에 대형점포가 수록되지 않아 제외 (별도 확보 예정)
}

# 코드 길이로 중분류/소분류를 구분한다 (4자리=중분류, 6자리=소분류)
def 업종파라미터(code):
    return {"indsMclsCd": code} if len(code) == 4 else {"indsSclsCd": code}

# 후보 15개가 걸쳐 있는 시군구 목록
후보시군구 = sorted(최종후보["행정구역명"].unique())
print(len(후보시군구), "개 시군구:", 후보시군구)


# ===== API 호출 캐시 =====
os.makedirs("data/cache", exist_ok=True)

def 캐시로드(경로, 수집함수):
    """파일이 있으면 읽어오고, 없을 때만 API를 호출해 수집한 뒤 저장한다.

    커널을 다시 띄울 때마다 수백 번씩 API를 부르지 않게 하려는 것.
    발표 전날 공공데이터포털이 느리거나 장애가 나도 노트북이 멈추지 않는다.
    다시 수집하려면 data/cache 폴더의 해당 파일을 지우면 된다.
    """
    if os.path.exists(경로):
        print(f"캐시 사용: {경로}")
        # TP_BUZ_NO·시도코드는 앞자리 0이 날아가지 않게 문자열로 고정
        return pd.read_csv(경로, encoding="utf-8-sig",
                           dtype={"TP_BUZ_NO": str, "시도코드": str})
    print(f"캐시 없음 → API 수집 시작: {경로}")
    df = 수집함수()
    df.to_csv(경로, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {경로}")
    return df

업소수 수집

In [ ]:
# 인구 데이터의 행정구역코드 앞 5자리 = 상가정보의 signguCd
코드표 = (pop[["행정구역명", "행정구역코드"]].drop_duplicates()
        .assign(sggCd=lambda d: d["행정구역코드"].str[:5])
        .set_index("행정구역명")["sggCd"])

API = "http://apis.data.go.kr/B553077/api/open/sdsc2/storeListInDong"

def 업소수(sggCd, 업종코드):
    """numOfRows=1로 요청해 totalCount만 읽는다 (데이터는 안 받아 가볍다)"""
    p = {"serviceKey": DATA_GO_KR_KEY, "divId": "signguCd", "key": sggCd,
         "numOfRows": "1", "pageNo": "1", "type": "json"}
    p.update(업종파라미터(업종코드))
    try:
        d = requests.get(API, params=p, timeout=30).json()
        # 해당 업종 점포가 0개면 NODATA_ERROR가 오고 totalCount가 없다
        return int(d.get("body", {}).get("totalCount") or 0)
    except Exception as e:
        # 예외 메시지에는 serviceKey가 박힌 URL이 들어있어 그대로 찍으면 키가 노출된다
        print("  실패:", sggCd, 업종코드, type(e).__name__)
        return None

def 수집_후보업소수():
    행 = []
    for 지역 in 후보시군구:
        sgg = 코드표[지역]
        for bc코드, 상가코드들 in 업종매핑.items():
            합 = 0
            for c in 상가코드들:
                n = 업소수(sgg, c)
                if n is None:          # 호출 실패 시 그 조합은 결측 처리
                    합 = None
                    break
                합 += n
                time.sleep(0.35)       # 공공 API에 부담 주지 않도록 간격
            행.append({"행정구역명": 지역, "TP_BUZ_NO": bc코드, "업소수": 합})
        print(f"  {지역} 완료")
    return pd.DataFrame(행)

업소 = 캐시로드("data/cache/업소수_후보시군구.csv", 수집_후보업소수)
print("수집 결과:", 업소.shape, "/ 결측:", 업소["업소수"].isna().sum())
display(업소.head(10))

대형마트 수집

In [ ]:
# 대규모점포는 전국 4,183건뿐이라 전체를 받아서 걸러 쓴다
대규모API = "https://apis.data.go.kr/1741000/large_scale_retail_stores/info"

def 수집_대규모점포():
    점포 = []
    for pg in range(1, 45):                      # 페이지당 100건 상한
        r = requests.get(대규모API, params={"serviceKey": DATA_GO_KR_KEY,
                                          "pageNo": pg, "numOfRows": 100, "type": "json"}, timeout=60)
        items = r.json()["response"]["body"]["items"]["item"]
        점포 += items
        if len(items) < 100:
            break
        time.sleep(0.25)
    return pd.DataFrame(점포)

대규모 = 캐시로드("data/cache/대규모점포_원본.csv", 수집_대규모점포)
print("전체:", len(대규모), "건")

# 영업 중인 대형마트만 (BC의 '대형할인점'에 대응, 백화점·쇼핑센터는 BC 업종에 없다)
마트 = 대규모[(대규모["SALS_STTS_NM"] == "영업/정상") &
            (대규모["BZSTAT_SE_NM"] == "대형마트")].copy()
print("영업 중 대형마트:", len(마트), "개")

In [ ]:
# 주소에서 시군구를 뽑아 우리 지역명과 맞춘다
주소 = 마트["LOTNO_ADDR"].fillna("") + " " + 마트["ROAD_NM_ADDR"].fillna("")

# 2026년 6~8월 행정구역 개편 보정
#  점포 데이터는 개편 후(전남광주통합특별시), BC·인구는 개편 전(광주광역시/전라남도) 체계다
#  통합특별시 아래 '구'는 광주, '시/군'은 전라남도로 되돌린다
주소 = (주소.str.replace(r"전남광주통합특별시(\s+[가-힣]+구)", r"광주광역시\1", regex=True)
          .str.replace("전남광주통합특별시", "전라남도", regex=False))

# 긴 이름(3단어 분구)부터 매칭해야 '수원시'가 '수원시 팔달구'를 가로채지 않는다
지역목록 = sorted(pop["행정구역명"].unique(), key=len, reverse=True)

def 지역찾기(a):
    for g in 지역목록:
        if g in a:
            return g
    return None

마트["행정구역명"] = 주소.apply(지역찾기)
print("매칭 실패:", 마트["행정구역명"].isna().sum(), "건 (인천 개편 지역 등)")

마트수 = 마트.groupby("행정구역명").size().rename("업소수").reset_index()
마트수["TP_BUZ_NO"] = "4004"          # 대형할인점

display(마트수[마트수["행정구역명"].isin(후보시군구)])


전국 기준선용 업소수 수집

In [ ]:
# 시도 코드는 행정구역코드 앞 2자리 (11 서울, 26 부산, 41 경기 …)
시도코드 = sorted(pop["행정구역코드"].astype(str).str[:2].unique())
print(len(시도코드), "개 시도:", 시도코드)

def 수집_전국업소수():
    전국행 = []
    for sido in 시도코드:
        for bc코드, 상가코드들 in 업종매핑.items():
            합 = 0
            for c in 상가코드들:
                p = {"serviceKey": DATA_GO_KR_KEY, "divId": "ctprvnCd", "key": sido,
                     "numOfRows": "1", "pageNo": "1", "type": "json"}
                p.update(업종파라미터(c))
                try:
                    합 += int(requests.get(API, params=p, timeout=30)
                             .json().get("body", {}).get("totalCount") or 0)
                except Exception as e:
                    # 예외 메시지에 키가 포함된 URL이 들어있어 예외 종류만 찍는다
                    print("  실패:", sido, c, type(e).__name__)
                time.sleep(0.35)
            전국행.append({"시도코드": sido, "TP_BUZ_NO": bc코드, "업소수": 합})
        print(f"  시도 {sido} 완료")
    # 시도별을 합쳐 업종별 전국 업소수로
    return (pd.DataFrame(전국행).groupby("TP_BUZ_NO", as_index=False)["업소수"].sum()
            .rename(columns={"업소수": "전국업소수"}))

전국업소 = 캐시로드("data/cache/업소수_전국.csv", 수집_전국업소수)

# 대형할인점(4004)은 상가정보에 없으므로 대규모점포 데이터에서 채운다
# (캐시로 읽어온 경우 이미 들어 있을 수 있어 중복 추가를 막는다)
if "4004" not in set(전국업소["TP_BUZ_NO"]):
    전국업소 = pd.concat([전국업소, pd.DataFrame([{
        "TP_BUZ_NO": "4004", "전국업소수": len(마트)      # 영업 중 대형마트 전국 합계
    }])], ignore_index=True)

display(전국업소)

 3단계 검증 — 업소당 BC 소비액

In [ ]:
# 1) 후보 지역 업소수 = 상가정보 8개 업종 + 대규모점포에서 온 대형할인점
업소전체 = pd.concat([
    업소,
    마트수[마트수["행정구역명"].isin(후보시군구)][["행정구역명", "TP_BUZ_NO", "업소수"]]
], ignore_index=True)

# 2) 후보 15개에 업종코드가 남아 있는 원본(저침투)에서 다시 뽑는다
후보상세 = 저침투.sort_values("종합점수").head(15).copy()

# 3) 업소수 붙이기
검증 = 후보상세.merge(업소전체, on=["행정구역명", "TP_BUZ_NO"], how="left")
print("업소수 결측:", 검증["업소수"].isna().sum(), "건")

# 4) 업종별 전국 기준선
#    전국 BC 소비(업종) ÷ 6개월 ÷ 전국 업소수 = 업소 1곳당 월평균 BC 소비
전국소비 = bc_mix.groupby("TP_BUZ_NO")["amt"].sum().rename("전국소비")
기준선 = (전국업소.merge(전국소비, on="TP_BUZ_NO")
        .assign(전국_업소당월소비=lambda d: d["전국소비"] / 6 / d["전국업소수"]))

display(기준선[["TP_BUZ_NO", "전국업소수", "전국_업소당월소비"]])


In [ ]:
# 5) 후보의 업소당 월 BC 소비 vs 전국 평균
검증 = 검증.merge(기준선[["TP_BUZ_NO", "전국_업소당월소비"]], on="TP_BUZ_NO", how="left")
검증["업소당월소비"] = 검증["amt"] / 6 / 검증["업소수"]

# 침투지수: 100 = 전국 평균만큼 BC가 쓰임, 낮을수록 점포는 있는데 BC 결제가 적다
검증["침투지수"] = 검증["업소당월소비"] / 검증["전국_업소당월소비"] * 100

결과 = (검증.sort_values("침투지수")
       .assign(금액_억=lambda d: d["amt"] / 1e8,
               업소당월소비_만원=lambda d: d["업소당월소비"] / 1e4,
               전국평균_만원=lambda d: d["전국_업소당월소비"] / 1e4)
       [["행정구역명", "TP_BUZ_NM", "금액_억", "업소수",
         "업소당월소비_만원", "전국평균_만원", "침투지수", "기대치대비%", "특화지수"]])

display(결과)


전국 기준선 보정

In [ ]:
# 상가정보 API에 광주광역시·전라남도가 누락돼 있다(행정구역 통합 작업 중으로 추정).
# 업소수 분모에 두 지역이 빠졌으므로, 분자인 BC 소비에서도 같이 빼야 정합적이다.
# 단 대형할인점(4004)은 업소수를 대규모점포 데이터(전국 전체)에서 받았으므로 보정하지 않는다.
누락지역 = ["광주광역시", "전라남도"]

소비_보정 = (bc_mix[~bc_mix["SIDO_NM"].isin(누락지역)]
           .groupby("TP_BUZ_NO")["amt"].sum().rename("전국소비"))
소비_전체 = bc_mix.groupby("TP_BUZ_NO")["amt"].sum().rename("전국소비")

# 업종별로 맞는 쪽을 고른다
전국소비2 = 소비_보정.copy()
전국소비2["4004"] = 소비_전체["4004"]          # 대형할인점만 전국 전체

기준선2 = (전국업소.merge(전국소비2, on="TP_BUZ_NO")
         .assign(전국_업소당월소비=lambda d: d["전국소비"] / 6 / d["전국업소수"]))

display(기준선2.assign(전국평균_만원=lambda d: d["전국_업소당월소비"] / 1e4)
        [["TP_BUZ_NO", "전국업소수", "전국평균_만원"]])


In [ ]:
# 광주 북구는 상가정보에 데이터가 없어 업소수 0 → 검증 불가로 분리
검증2 = 검증.drop(columns=["전국_업소당월소비"]).merge(
    기준선2[["TP_BUZ_NO", "전국_업소당월소비"]], on="TP_BUZ_NO", how="left")

검증불가 = 검증2[검증2["업소수"] == 0].copy()
검증2 = 검증2[검증2["업소수"] > 0].copy()

검증2["업소당월소비"] = 검증2["amt"] / 6 / 검증2["업소수"]
검증2["침투지수"] = 검증2["업소당월소비"] / 검증2["전국_업소당월소비"] * 100

# 등급 분류 — 3단계의 최종 판정
def 등급(x):
    if x < 70:   return "1_확실한 저침투"
    if x < 90:   return "2_경계"
    return "3_탈락(시장이 작았던 것)"
검증2["판정"] = 검증2["침투지수"].apply(등급)

결과2 = (검증2.sort_values("침투지수")
        .assign(금액_억=lambda d: d["amt"] / 1e8,
                업소당월소비_만원=lambda d: d["업소당월소비"] / 1e4,
                전국평균_만원=lambda d: d["전국_업소당월소비"] / 1e4)
        [["판정", "행정구역명", "TP_BUZ_NM", "금액_억", "업소수",
          "업소당월소비_만원", "전국평균_만원", "침투지수"]])

display(결과2)
print()
print("검증 불가(상가정보 누락):")
display(검증불가[["행정구역명", "TP_BUZ_NM"]])


외국인 포함 재계산 (3단계 최종)

In [ ]:
# 기획서 3단계 지침: "외국인 소비는 포함해서 계산한다
#  (상권 총매출 자체가 내·외국인 구분 없이 집계되므로 BC 쪽 분자도 범위를 맞춰야 함)"
# 분모인 업소수가 내·외국인 구분 없는 전체 점포이므로 분자도 맞춘다.
# 연령 제한(20대 이상)도 푼다 — 그건 인구를 분모로 쓸 때 필요했던 보정이다.
bc_검증 = bc[bc["GENDER_CD"].isin(["1", "2", "3"])]      # 내국인 + 외국인, 법인만 제외

# 후보 지역 분자
소비_후보 = (bc_검증.groupby(["행정구역명", "TP_BUZ_NO"], as_index=False)["amt"]
           .sum().rename(columns={"amt": "amt검증"}))

# 전국 기준선 (광주·전남 보정은 그대로 유지, 대형할인점만 예외)
소비_보정2 = bc_검증[~bc_검증["SIDO_NM"].isin(누락지역)].groupby("TP_BUZ_NO")["amt"].sum()
소비_전체2 = bc_검증.groupby("TP_BUZ_NO")["amt"].sum()
전국소비3 = 소비_보정2.copy()
전국소비3["4004"] = 소비_전체2["4004"]

기준선3 = (전국업소.merge(전국소비3.rename("전국소비"), on="TP_BUZ_NO")
         .assign(전국_업소당월소비=lambda d: d["전국소비"] / 6 / d["전국업소수"]))

# 최종 계산
최종 = (검증2[["행정구역명", "TP_BUZ_NO", "TP_BUZ_NM", "업소수"]]
       .merge(소비_후보, on=["행정구역명", "TP_BUZ_NO"])
       .merge(기준선3[["TP_BUZ_NO", "전국_업소당월소비"]], on="TP_BUZ_NO"))

최종["업소당월소비"] = 최종["amt검증"] / 6 / 최종["업소수"]
최종["침투지수"] = 최종["업소당월소비"] / 최종["전국_업소당월소비"] * 100
최종["판정"] = 최종["침투지수"].apply(등급)

display(최종.sort_values("침투지수")
        .assign(금액_억=lambda d: d["amt검증"] / 1e8)
        [["판정", "행정구역명", "TP_BUZ_NM", "금액_억", "업소수", "침투지수"]])


전국 시군구 업소수 수집

In [ ]:
# 광주광역시(29)·전라남도(46)는 상가정보 API에 데이터가 없다.
# 0으로 잡으면 "점포가 하나도 없는 극단적 저침투"로 오인되므로 아예 수집 대상에서 뺀다.
누락시도 = ["29", "46"]

전체코드표 = (pop[["행정구역명", "행정구역코드"]].drop_duplicates()
           .assign(sggCd=lambda d: d["행정구역코드"].str[:5],
                   sidoCd=lambda d: d["행정구역코드"].str[:2]))
수집대상 = 전체코드표[~전체코드표["sidoCd"].isin(누락시도)].sort_values("행정구역명")

print(f"수집 대상 {len(수집대상)}개 지역 (전체 {len(전체코드표)}개 중 광주·전남 제외)")
print(f"예상 호출 {len(수집대상) * 15:,}회 / 약 20분")


In [ ]:
부분저장 = "data/cache/업소수_전국시군구_부분.csv"

def 수집_전국시군구업소수():
    # 이미 받아둔 후보 14곳을 재사용하고, 중단됐으면 거기서부터 이어받는다
    if os.path.exists(부분저장):
        기존 = pd.read_csv(부분저장, encoding="utf-8-sig", dtype={"TP_BUZ_NO": str})
        print(f"  이어받기: {기존['행정구역명'].nunique()}개 지역 완료 상태")
    else:
        기존 = pd.read_csv("data/cache/업소수_후보시군구.csv",
                         encoding="utf-8-sig", dtype={"TP_BUZ_NO": str})
        print(f"  후보 캐시 재사용: {기존['행정구역명'].nunique()}개 지역 건너뜀")

    행 = 기존.to_dict("records")
    done = set(기존["행정구역명"])
    남은 = [r for _, r in 수집대상.iterrows() if r["행정구역명"] not in done]
    print(f"  남은 지역 {len(남은)}개")

    t0 = time.time()
    for i, r in enumerate(남은, 1):
        for bc코드, 상가코드들 in 업종매핑.items():
            합 = 0
            for c in 상가코드들:
                n = 업소수(r["sggCd"], c)
                합 += (n or 0)       # 실패·NODATA는 0으로
                time.sleep(0.3)
            행.append({"행정구역명": r["행정구역명"], "TP_BUZ_NO": bc코드, "업소수": 합})

        # 20개 지역마다 중간 저장 — 끊겨도 여기서부터 다시 시작한다
        if i % 20 == 0 or i == len(남은):
            pd.DataFrame(행).to_csv(부분저장, index=False, encoding="utf-8-sig")
            경과 = time.time() - t0
            남은시간 = 경과 / i * (len(남은) - i)
            print(f"  {i}/{len(남은)}  경과 {경과/60:.1f}분  남은 예상 {남은시간/60:.1f}분")

    return pd.DataFrame(행)

전국시군구업소 = 캐시로드("data/cache/업소수_전국시군구.csv", 수집_전국시군구업소수)

print()
print("결과:", 전국시군구업소.shape)
print("지역 수:", 전국시군구업소["행정구역명"].nunique())
display(전국시군구업소.groupby("TP_BUZ_NO")["업소수"].sum().rename("전국 합계"))


수집 결과 보정

In [ ]:
# 1) 광주광역시·전라남도는 상가정보 API에 데이터가 없다.
#    후보 캐시를 재사용하면서 광주 북구가 0으로 끌려 들어왔으므로 제거한다.
전국시군구업소 = 전국시군구업소[
    ~전국시군구업소["행정구역명"].str.startswith(("광주광역시", "전라남도"))
].copy()

# 2) 개별 실패분 재수집
#    - 대전 서구 슈퍼마켓 / 대전 동구 서양음식 : 수집 중 타임아웃
#    - 전북 진안군 일식회집 : 에러 없이 0이 들어옴 (API 일시적 빈 응답)
재수집 = [("대전광역시 서구", "30170", "4020"),
        ("대전광역시 동구", "30110", "8006"),
        ("전북특별자치도 진안군", "52750", "8004")]

for 지역, sgg, bc코드 in 재수집:
    합 = sum(업소수(sgg, c) or 0 for c in 업종매핑[bc코드])
    전국시군구업소.loc[(전국시군구업소["행정구역명"] == 지역) &
                  (전국시군구업소["TP_BUZ_NO"] == bc코드), "업소수"] = 합
    print(f"  {지역} {bc코드} → {합}")
    time.sleep(0.3)


In [ ]:
# 3) 인천 행정구역 개편 보정
#    상가정보는 개편 후 체계, BC·인구는 개편 전 체계라 코드가 안 맞아 0이 나왔다.
#      중구 + 동구  ↔  제물포구 + 영종구
#      서구         ↔  서해구 + 검단구
#    1:1이 아니라 합집합끼리만 일치하므로, 합쳐서 하나의 분석 단위로 다룬다.
인천보정 = {
    "인천광역시 중구+동구": ["28125", "28155"],   # 제물포구, 영종구
    "인천광역시 서구":      ["28275", "28290"],   # 서해구, 검단구
}

행 = []
for 지역, 코드들 in 인천보정.items():
    for bc코드, 상가코드들 in 업종매핑.items():
        합 = 0
        for sgg in 코드들:
            for c in 상가코드들:
                합 += (업소수(sgg, c) or 0)
                time.sleep(0.3)
        행.append({"행정구역명": 지역, "TP_BUZ_NO": bc코드, "업소수": 합})
    print(f"  {지역} 완료")

# 0으로 채워진 옛 체계 행을 버리고 보정값으로 교체
전국시군구업소 = pd.concat([
    전국시군구업소[~전국시군구업소["행정구역명"].isin(
        ["인천광역시 중구", "인천광역시 동구", "인천광역시 서구"])],
    pd.DataFrame(행)
], ignore_index=True)

# 저장해서 다음부터는 다시 안 받게 한다
전국시군구업소.to_csv("data/cache/업소수_전국시군구.csv", index=False, encoding="utf-8-sig")

print()
print("지역 수:", 전국시군구업소["행정구역명"].nunique())
print("업소수 0인 조합:", (전국시군구업소["업소수"] == 0).sum(), "건 (군위군 일식만 남으면 정상)")
display(전국시군구업소[전국시군구업소["업소수"] == 0])


대형할인점 붙이고 BC·인구 쪽 인천 병합

In [ ]:
# 1) 대형할인점(4004) — 이미 받아둔 대규모점포 데이터에서 지역별로 센다 (API 호출 없음)
마트수_전국 = 마트.groupby("행정구역명").size().rename("업소수").reset_index()
마트수_전국["TP_BUZ_NO"] = "4004"

# 2) 인천 개편 보정을 마트 쪽에도 적용
#    대규모점포 데이터는 2026년 8월 기준이라 이미 신 체계(제물포·영종·서해·검단)를 쓴다
인천신구 = {"인천광역시 제물포구": "인천광역시 중구+동구",
          "인천광역시 영종구":   "인천광역시 중구+동구",
          "인천광역시 서해구":   "인천광역시 서구",
          "인천광역시 검단구":   "인천광역시 서구",
          "인천광역시 중구":     "인천광역시 중구+동구",
          "인천광역시 동구":     "인천광역시 중구+동구"}
마트수_전국["행정구역명"] = 마트수_전국["행정구역명"].replace(인천신구)
마트수_전국 = 마트수_전국.groupby(["행정구역명", "TP_BUZ_NO"], as_index=False)["업소수"].sum()

# 3) 9개 업종 업소수 완성
업소_전국 = pd.concat([전국시군구업소, 마트수_전국], ignore_index=True)

print("업소수 데이터:", 업소_전국.shape)
print("지역 수:", 업소_전국["행정구역명"].nunique())


In [ ]:
# BC·인구 쪽도 인천 중구+동구를 하나로 합친다 (업소수와 단위를 맞추기 위해)
def 지역병합(df):
    d = df.copy()
    d["행정구역명"] = d["행정구역명"].replace(
        {"인천광역시 중구": "인천광역시 중구+동구",
         "인천광역시 동구": "인천광역시 중구+동구"})
    return d

# 분자: 외국인 포함, 법인 제외 (3단계와 동일 기준)
BC소비 = (지역병합(bc[bc["GENDER_CD"].isin(["1", "2", "3"])])
        .groupby(["행정구역명", "TP_BUZ_NO"], as_index=False)["amt"].sum())

# 인구: 성인(20대 이상) 연령대별 — 회귀 설명변수로 쓴다
인구_지역 = (지역병합(pop[pop["AGE_CD"].isin(성인)])
          .groupby(["행정구역명", "AGE_CD", "STRD_YYMM"])["인구"].sum()
          .groupby(["행정구역명", "AGE_CD"]).mean()      # 6개월 평균
          .unstack())
인구_지역["성인인구"] = 인구_지역.sum(axis=1)
for a in 성인:
    인구_지역[f"비율_{a}"] = 인구_지역[a] / 인구_지역["성인인구"]

print("BC 소비:", BC소비.shape, "/ 인구:", 인구_지역.shape)


In [ ]:
# 세 데이터를 합쳐 회귀용 테이블 완성
회귀데이터 = (업소_전국
          .merge(BC소비, on=["행정구역명", "TP_BUZ_NO"], how="inner")
          .merge(인구_지역.reset_index(), on="행정구역명", how="inner"))

# 업소수 0 또는 소비 0이면 로그를 못 씌우므로 제외
회귀데이터 = 회귀데이터[(회귀데이터["업소수"] > 0) & (회귀데이터["amt"] > 0)].copy()

print("회귀용 조합:", len(회귀데이터))
print("지역:", 회귀데이터["행정구역명"].nunique(), "/ 업종:", 회귀데이터["TP_BUZ_NO"].nunique())
display(회귀데이터.groupby("TP_BUZ_NO").size().rename("표본수"))


업종별 회귀

In [ ]:
import numpy as np
import statsmodels.api as sm

# 로그 변환 — 금액·업소수·인구 모두 오른쪽 꼬리가 긴 분포라 로그를 씌워야 등분산 가정이 맞는다
# 계수 해석도 "업소수 1% 증가 시 소비 몇 % 증가"로 자연스러워진다
회귀데이터["log_amt"] = np.log(회귀데이터["amt"])
회귀데이터["log_업소수"] = np.log(회귀데이터["업소수"])
회귀데이터["log_인구"] = np.log(회귀데이터["성인인구"])

# 연령 비율 5개는 합이 1이라 전부 넣으면 완전공선성이 생긴다.
# 40대(비율_4)를 기준으로 빼고 나머지 4개만 넣는다.
설명변수 = ["log_업소수", "log_인구", "비율_2", "비율_3", "비율_5", "비율_6"]

결과행, 잔차모음 = [], []
for 업종코드, d in 회귀데이터.groupby("TP_BUZ_NO"):
    X = sm.add_constant(d[설명변수])
    모델 = sm.OLS(d["log_amt"], X).fit()

    # 잔차 = 실제 - 예측. 로그 공간이라 exp를 씌우면 "예측 대비 몇 배"가 된다
    잔차모음.append(d.assign(잔차=모델.resid, 회귀지수=np.exp(모델.resid) * 100))

    결과행.append({"TP_BUZ_NO": 업종코드, "n": len(d), "R2": 모델.rsquared,
                 "log_업소수": 모델.params["log_업소수"], "p_업소수": 모델.pvalues["log_업소수"],
                 "log_인구": 모델.params["log_인구"], "p_인구": 모델.pvalues["log_인구"]})

회귀결과 = pd.concat(잔차모음, ignore_index=True)
display(pd.DataFrame(결과행).round(3))


In [ ]:
# 업종코드 → 이름 매핑 (회귀데이터에는 코드만 있어서 붙여 준다)
업종명 = (bc[["TP_BUZ_NO", "TP_BUZ_NM"]].drop_duplicates()
        .set_index("TP_BUZ_NO")["TP_BUZ_NM"])
회귀결과["TP_BUZ_NM"] = 회귀결과["TP_BUZ_NO"].map(업종명)

# 시장규모 필터 — 금액 0억대 시골이 상위를 차지하는 걸 막는다
회귀결과["업종내중앙값"] = 회귀결과.groupby("TP_BUZ_NO")["amt"].transform("median")
전국후보 = 회귀결과[회귀결과["amt"] >= 회귀결과["업종내중앙값"]].copy()

print(f"필터 전 {len(회귀결과)} → 업종 내 상위 50%: {len(전국후보)}")

display(전국후보.nsmallest(20, "회귀지수")
        .assign(금액_억=lambda d: d["amt"] / 1e8)
        [["행정구역명", "TP_BUZ_NM", "금액_억", "업소수", "회귀지수"]])


In [ ]:
# 지역 단위 종합 — 업종 8개 이상 있는 지역만
지역종합 = (회귀결과.groupby("행정구역명")
         .agg(업종수=("회귀지수", "size"), 평균지수=("회귀지수", "mean"),
              백미만=("회귀지수", lambda s: (s < 100).sum()), 총액=("amt", "sum")))
지역종합 = 지역종합[지역종합["업종수"] >= 8]

display(지역종합.nsmallest(15, "평균지수")
        .assign(총액_억=lambda d: d["총액"] / 1e8)
        [["업종수", "평균지수", "백미만", "총액_억"]])


In [ ]:
# 지역 단위 종합 — 업종 8개 이상 있는 지역만
지역종합 = (회귀결과.groupby("행정구역명")
         .agg(업종수=("회귀지수", "size"), 평균지수=("회귀지수", "mean"),
              백미만=("회귀지수", lambda s: (s < 100).sum()), 총액=("amt", "sum")))
지역종합 = 지역종합[지역종합["업종수"] >= 8]

display(지역종합.nsmallest(15, "평균지수")
        .assign(총액_억=lambda d: d["총액"] / 1e8)
        [["업종수", "평균지수", "백미만", "총액_억"]])
